# 从零开始用代码实现 GPT - 练习解答

来自 [gpt, from scratch 视频](https://www.youtube.com/watch?v=kCc8FmEb1nY)的练习解答笔记。<br>
欢迎先尝试自己解决，使用[这个入门笔记本](./N007%20-%20GPT_Exercises.ipynb)。

1. 在 YouTube 上观看 [gpt, from scratch 视频](https://www.youtube.com/watch?v=kCc8FmEb1nY)
2. 回来解决这些练习以提升技能 :)

*强烈*建议使用带 GPU 的机器来完成这些练习。

In [2]:
import torch
import random
import torch.nn as nn
from tqdm import tqdm
from torch.nn import functional as F

## 练习 1 - $n$ 维张量（tensor）掌握挑战

**目标：** 将 `Head` 和 `MultiHeadAttention` 合并为一个类，该类并行处理所有注意力头，<br>
将头作为另一个批次维度来处理（答案也可以在 [nanoGPT](https://github.com/karpathy/nanoGPT) 中找到）。

让我们看看我们要处理的内容：

In [2]:
block_size = 256 # What is the maximum context length for predictions?
dropout = 0.2    # Dropout probability
n_embd = 384     # Number of hidden units in the Transformer (384/6 = 64 dimensions per head)

In [3]:
class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Register a buffer so that it is not a parameter of the model
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape   # Batch size, block size, vocab size (each token is a vector of size 32)
        k = self.key(x)   # (B,T,C) -> (B,T, head_size)
        q = self.query(x) # (B,T,C) -> (B,T, head_size)
        # Compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5                       # (B, T, head_size) @ (B, head_size, T) = (B, T, T) (T is the block_size)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # Masking all values in wei where tril == 0 with -inf
        wei = F.softmax(wei, dim=-1)                                 # (B, T, T)
        wei = self.dropout(wei)
        # Weighted aggregation of the values
        v = self.value(x) # (B, T, C) -> (B, T, head_size)
        out = wei @ v     # (B, T, T) @ (B, T, head_size) = (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) # Create num_heads many heads
        self.proj = nn.Linear(n_embd, n_embd)                                   # Projecting back to n_embd dimensions (the original size of the input, because we use residual connections)
        self.dropout = nn.Dropout(dropout)                                      # Dropout layer for regularization

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # Concatenate the outputs of all heads
        out = self.dropout(self.proj(out))                  # Project back to n_embd dimensions (because we use residual connections) and apply dropout
        return out

我们希望 `key`、`query` 和 `value` 线性层能够跨所有 `num_heads` 并行应用。

首先，回想一下每个 token 由一个大小为 `n_embd=384` 的向量表示。<br>
多头注意力（multi-head attention）将这个 `n_embd` 分配到更小的、大小相等的头上。<br>
在这种情况下，`head_size` 是每个头从总嵌入中接收的部分。

我们可以将内部 `n_embd` 灵活地写为 `num_heads` 和 `head_size` 的乘积。<br>
这样可以更好地泛化到任意数量的头和头大小。

下面是实现的融合 `Head` 和 `MultiHeadAttention` 类，附有注释引导：

In [4]:
class MultiHeadAttention(nn.Module):
    """ Multi-head self-attention processing all heads in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.num_heads = num_heads           # Apply this many parallel attention layers
        self.head_size = head_size           # Each head has this size (part of the embedding size)
        self.n_embd = num_heads * head_size  # Total size of all heads together forms the token embedding size

        # Combining key, query, and value transformations across heads in a single linear layer each
        # All heads together process the input sequence and all together produce the output sequence
        # As self.embed = num_heads * head_size, input and output dim for all heads at once are the same (n_embd)
        self.key = nn.Linear(self.n_embd, self.n_embd, bias=False)
        self.query = nn.Linear(self.n_embd, self.n_embd, bias=False)
        self.value = nn.Linear(self.n_embd, self.n_embd, bias=False)

        # Register a buffer so that causal mask is not a parameter of the model
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        # Final linear output transformation, dropout
        # Same as with the key, query, value transformations,
        # As self.embed = num_heads * head_size, input and output dim for all heads at once are the same (n_embd)
        self.proj = nn.Linear(self.n_embd, self.n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # Batch size, sequence length (aka block size), embedding size (aka vocab size)

        # Apply linear transformations to get keys, queries, and values for all heads
        # Produce a dimension for the number of heads and a dimension for the head size
        # We then move the T dimension to the second index position make the attention matrix multiplication applicable
        k = self.key(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)    # (B, T, C) -> (B, num_heads, T, head_size)
        q = self.query(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)  # (B, T, C) -> (B, num_heads, T, head_size)
        v = self.value(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)  # (B, T, C) -> (B, num_heads, T, head_size)

        # Compute the attention scores
        wei = q @ k.transpose(-2, -1) * self.head_size ** -0.5        # (B, num_heads, T, head_size) @ (B, num_heads, head_size, T) -> (B, num_heads, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # Apply the causal mask, i.e. mask out the upper triangular part of the attention matrix
        wei = F.softmax(wei, dim=-1)  # Normalize attention scores to form (pseudo-)probabilities
        wei = self.dropout(wei)       # Apply dropout, promotes flexibility and robustness

        # Weighted aggregation of values
        out = wei @ v  # (B, num_heads, T, T) @ (B, num_heads, T, head_size) -> (B, num_heads, T, head_size)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, num_heads, T, head_size) -> (B, T, C)

        # Final projection
        out = self.dropout(self.proj(out))
        return out

我现在将这个整合到视频衍生的 GPT 实现中，并首先在 `tiny-shakespeare.txt` 数据集上运行以验证实现并为后续练习生成基线：

In [ ]:
# Hyperparameters
batch_size = 64      # How many independent sequences to process at once?
block_size = 256     # What is the maximum context length for predictions?
max_iters = 5000     # How many training iterations to run?
eval_interval = 500  # How often to evaluate the model on the validation set?
learning_rate = 3e-4 # Learning rate for Adam optimizer (found through trial and error)
device = 'cuda' if torch.cuda.is_available() else 'cpu' # Don't run on CPU if possible (it's slow. really.)
eval_iters = 200     # How many batches to use per loss evaluation?
n_embd = 384         # Number of hidden units in the Transformer (384/6 = 64 dimensions per head)
n_head = 6           # Number of attention heads in a single Transformer layer
n_layer = 6          # Number of Transformer layers
dropout = 0.2        # Dropout probability

torch.manual_seed(1337)
print(f'Training on {device}')

# Load Tiny Shakespeare dataset
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# (also refer to Andrej's blog: http://karpathy.github.io/2015/05/21/rnn-effectiveness/)
with open('../tiny-shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Find all unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Create mappings from characters to indices and vice versa
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]          # encoder: Take a string, return a list of indices/integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: Take a list of indices/integers, return a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)) # first 90% of all characters are for training
train_data = data[:n]
val_data = data[n:]

# Data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # Generates a tensor of shape (batch_size,) with random sequence start indices between 0 and len(data) - block_size
    x = torch.stack([data[i:i+block_size] for i in ix])       # Stack all (ix holds batch_size many) sequences of this batch row-wise on top of each other to form a tensor
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])   # Same as x but shifted by one token
    x, y = x.to(device), y.to(device)
    return x, y # x is batch_size x block_size, y is batch_size x block_size

@torch.no_grad() # Disable gradient calculation for this function
def estimate_loss():
    out = {}
    model.eval() # Set model to evaluation mode
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # Set model back to training mode
    return out

class MultiHeadAttention(nn.Module):
    """ Multi-head self-attention processing all heads in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.num_heads = num_heads           # Apply this many parallel attention layers
        self.head_size = head_size           # Each head has this size (part of the embedding size)
        self.n_embd = num_heads * head_size  # Total size of all heads together forms the token embedding size

        # Combining key, query, and value transformations across heads in a single linear layer each
        # All heads together process the input sequence and all together produce the output sequence
        # As self.embed = num_heads * head_size, input and output dim for all heads at once are the same (n_embd)
        self.key = nn.Linear(self.n_embd, self.n_embd, bias=False)
        self.query = nn.Linear(self.n_embd, self.n_embd, bias=False)
        self.value = nn.Linear(self.n_embd, self.n_embd, bias=False)

        # Register a buffer so that causal mask is not a parameter of the model
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        # Final linear output transformation, dropout
        # Same as with the key, query, value transformations,
        # As self.embed = num_heads * head_size, input and output dim for all heads at once are the same (n_embd)
        self.proj = nn.Linear(self.n_embd, self.n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # Batch size, sequence length, embedding size

        # Apply linear transformations to get keys, queries, and values for all heads
        k = self.key(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)    # (B, T, C) -> (B, num_heads, T, head_size)
        q = self.query(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)  # (B, T, C) -> (B, num_heads, T, head_size)
        v = self.value(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)  # (B, T, C) -> (B, num_heads, T, head_size)

        # Compute the attention scores
        wei = q @ k.transpose(-2, -1) * self.head_size ** -0.5        # (B, num_heads, T, head_size) @ (B, num_heads, head_size, T) -> (B, num_heads, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # Apply the causal mask
        wei = F.softmax(wei, dim=-1)  # Normalize attention scores to form (pseudo-)probabilities
        wei = self.dropout(wei)       # Apply dropout, promotes flexibility and robustness

        # Weighted aggregation of values
        out = wei @ v  # (B, num_heads, T, T) @ (B, num_heads, T, head_size) -> (B, num_heads, T, head_size)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, num_heads, T, head_size) -> (B, T, C)

        # Final projection
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # Linear layer with 4*n_embd outputs (AIAYN suggests 4*n_embd for residual connections as channel size)
            nn.ReLU(),                     # ReLU introduces non-linearity
            nn.Linear(4 * n_embd, n_embd), # Linear layer with n_embd outputs
            nn.Dropout(dropout),           # Dropout layer for regularization
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head                    # Adapting the head size to the number of heads
        self.sa = MultiHeadAttention(n_head, head_size) # Self-attention multi-head layer (the communication)
        self.ffwd = FeedFoward(n_embd)                  # Feed-forward so that the output has the same dimension as the input (the computation)
        self.ln1 = nn.LayerNorm(n_embd)                 # Layer normalization (normalizes the output of the self-attention layer)
        self.ln2 = nn.LayerNorm(n_embd)                 # Layer normalization (normalizes the output of the feed-forward layer)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))                    # Residual connection, forking off to the self-attention layer, LayerNorm is applied before the self-attention layer
        x = x + self.ffwd(self.ln2(x))                  # Residual connection, forking off to the feed-forward layer, LayerNorm is again applied before the feed-forward layer
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embd = nn.Embedding(vocab_size, n_embd)                                   # Embedding the vocabulary, each individual token is represented by a vector of size vocab_size x n_embd
        self.position_embd = nn.Embedding(block_size, n_embd)                                # Embedding the position, each position is represented by a vector of size block_size x n_embd
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)                                         # Linear layer to map the embedding to the vocabulary size

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_embd = self.token_embd(idx)                               # Embedding the input, shape is (batch_size, block_size, n_embd) (B, T, n_embd)
        pos_embd = self.position_embd(torch.arange(T, device=device)) # Embedding the position by providing an integer sequence up to block_size, shape is (block_size, n_embd) (T, n_embd)
        x = tok_embd + pos_embd                                       # Adding the token embedding and the position embedding, shape is (batch_size, block_size, n_embd) (B, T, n_embd)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                      # Calculating the logits, shape is (batch_size, block_size, vocab_size) (B, T, C)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)            # Transpose logits to (B, C, T) (B=batch_size, T=block_size, C=vocab_size)
            targets = targets.view(B*T)             # Transpose targets to (B, T)
            loss = F.cross_entropy(logits, targets) # Calculating cross entropy loss across all tokens in the batch

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]                    # Condition on the last block_size tokens (B, T)
            logits, _ = self(idx_cond)                         # Forward pass (this is the forward function) with the current sequence of characters idx, results in (B, T, C)
            logits = logits[:, -1, :]                          # Focus on the last token from the logits (B, T, C) -> (B, C)
            probs = F.softmax(logits, dim=-1)                  # Calculate the set of probabilities for the next token based on this last token, results in (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # Sample the next token (B, 1), the token with the highest probability is sampled most likely
            idx = torch.cat((idx, idx_next), dim=1)            # Add the new token to the sequence (B, T+1) for the next iteration
        return idx

# Model
model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters') # print the number of parameters in the model

# Create a PyTorch optimizer
opt = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training loop
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')     # Get batch
    logits, loss = model(xb, yb)    # Forward pass
    opt.zero_grad(set_to_none=True) # Reset gradients
    loss.backward()                 # Backward pass
    opt.step()                      # Update parameters

torch.save(model, "gpt_tinyshakespeare.pt")

# Generate text from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)     # Start with single token as context
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

Training on cuda
10.788929 M parameters
step 0: train loss 4.2837, val loss 4.2825
step 500: train loss 1.8859, val loss 1.9993
step 1000: train loss 1.5367, val loss 1.7271
step 1500: train loss 1.3955, val loss 1.6106
step 2000: train loss 1.3112, val loss 1.5488
step 2500: train loss 1.2523, val loss 1.5206
step 3000: train loss 1.2033, val loss 1.4990
step 3500: train loss 1.1641, val loss 1.4893
step 4000: train loss 1.1263, val loss 1.4817
step 4500: train loss 1.0898, val loss 1.4814
step 4999: train loss 1.0560, val loss 1.5007

KING RICHARD II:
Now from sentence come throw and mercy.
Be yonder that I behold to do tower:
'Tis you be longed on our words to much urge.
Never know, with use boody to make and tweMILet him.
To chain the people mother, the rest veril
Let him bring God hence, but be afflied,
Were traitors with one heart the flower days:
The sarch of a companion, dost thou body this heir?
The again all-as this body's world;
That nightly he should be the bal deform'd,
No

听起来差不多。

## 练习 2 - 数学掌握

**目标：** 在你自己选择的数据集上训练 GPT！还有哪些数据可能很有趣？<br>
一个有趣的高级建议：训练 GPT 做两个数字的加法，即 $a+b=c$。一旦你做到了，进阶项目：用 GPT 构建一个计算器克隆，涵盖所有 $+-*/$ 运算。<br>
- 你可能会发现按逆序预测 $c$ 的数字很有帮助，因为典型的加法算法（你希望它学到的）也会从右到左进行。
- 你可能想要修改数据加载器以直接提供随机问题并跳过 `train.bin`、`val.bin` 的生成。<br>
- 你可能想要使用目标中的 $y=-1$ 来屏蔽仅指定问题的 $a+b$ 输入位置的损失（参见 CrossEntropyLoss 的 ignore_index）。你的 Transformer 学会加法了吗？一旦你做到了，进阶项目：用 GPT 构建一个计算器克隆，涵盖所有 $+-*/$ 运算。

**这不是一个简单的问题。** 但是，[GPT 可以在没有计算器的情况下解决数学问题](https://arxiv.org/abs/2309.03241)。<br>
你可能需要[思维链（Chain of Thought）](https://arxiv.org/abs/2412.14135)和其他[稍微更高级的架构](https://github.com/deepseek-ai/DeepSeek-V3/blob/main/DeepSeek_V3.pdf)追踪，但不要过度思考。

我将这个实现导出到一个单独的脚本中，以便在更大的生成问题数据集上更容易运行。<br>
你可以在 [`N007_GPT_Solved_Exercise_Mathematica.py`](./N007_GPT_Solved_Exercise_Mathematica.py) 文件中找到该脚本。我在单个 GPU 上训练了约 $10$ 分钟。

训练好的模型可以在这里找到：[nanogpt_mathematica](https://huggingface.co/Marcus2112/nanogpt_mathematica)。

值得注意的特点和功能：
- `block_size` 减少到 $27$，以最多容纳单个生成数学问题的最大长度。
- `max_iters` 增加到 $20,000$，`eval_interval` 扩展到 $1,000$。
- `learning_rate` 增加到 `1e-3`，并使用 `CosineAnnealingLR` 在训练过程中衰减到 `1e-4`。
- `n_head` 和 `n_layer` 都从 $6$ 减少到 $4$，主要是为了加速训练。
- `dropout` 减少到 $0.05$，以减轻遗忘过多但仍对模型进行一些正则化。
- `generate_problem` 实现了按需生成问题，用于训练和验证的批次组装。
- `generate_problem` 在训练过程中逐步平滑地引入运算 `+`、`-`、`*`、`/`。
- `generate_problem` 引入 `=` 字符来表示问题陈述的结束，引入 `;` 字符来表示解答陈述的结束。
- `generate_problem` 引入了基于加法的思维链来处理乘法，问题格式为 `2*3=[2+2+2=6]=6;`，帮助模型将乘法理解为重复加法。
- 结果*不是*按逆序预测的。
- 填充字符 `#` 作为 `ignore_index` 传递给 `F.cross_entropy` 以在损失计算中忽略。
- 推理部分提供了一组数学问题，模型负责解决它们。
- 模型输出在第一次出现 `;` 字符时被截断。

训练好的模型产生了以下推理输出：
```
Input: 2+3=
Generated: 2+3=3=5]=3
Expected: 5

Input: 5-2=
Generated: 5-2=2=3]=2
Expected: 3

Input: 4*3=
Generated: 4*3=4+4=12]=1=2222]]122818
Expected: 12

Input: 8/4=
Generated: 8/4=/4=/]=2
Expected: 2
```

模型始终将最终结果放在最后一个 `=` 之后、`]` 之前，或者如果没有 `=` 则作为终止数字。<br>
模型仍然有些胡言乱语，但结果表明在所有四种运算上都有明显的学习效果。<br>
这个结论得到了精心设计的示例输入序列的支持：<br>
我使示例中的输入序列不包含也属于预期解答的数字。<br>
然而关键是，模型能够在输出中生成解答数字。

该模型包含 $7.1213\ \text{M}$ 个参数，初始损失为 $3.3212$，训练结束时最终损失为 $0.0000$。

## 练习 3 - 微调（Finetuning）能做得更好吗？

**目标：** 找到一个非常大的数据集，大到你看不到训练损失和验证损失之间的差距。<br>
在这个数据上预训练 Transformer。然后，用该模型初始化并在 `tiny shakespeare` 上以更少的步数和更低的学习率进行微调。<br>你能通过大规模预训练获得更低的验证损失吗？

我将使用 Hugging Face 上的 [minipile_density-proportioned](https://huggingface.co/datasets/Marcus2112/minipile_density-proportioned) 数据集。<br>
它有 $946\text{k}$ 个文本示例和 $2$ 个特征，`text` 和 `pile_idx`。我们只需要 `text` 特征。

我将这个练习的代码导出到一个单独的脚本中，以便在更大的数据集上更容易运行。<br>
你可以在 [`N007_GPT_Solved_Exercise_Finetune.py`](./N007_GPT_Solved_Exercise_Finetune.py) 文件中找到该脚本。

预训练模型可以在这里找到：[nanogpt_base](https://huggingface.co/Marcus2112/nanogpt_base)。<br>
微调后的模型可以在这里找到：[nanogpt_shakespeare](https://huggingface.co/Marcus2112/nanogpt_shakespeare)。

**为什么选择这个数据集？**

我基于 [\[Kaddour, Jean. 2023\]](https://arxiv.org/abs/2304.08442) 自己组装了上述数据集。该论文提出了一种创建蒸馏数据集的方法，当用于训练时，能够使模型达到与在约 $100 \times$ 更大数据集上训练的模型相当的性能。类似于论文的做法，我通过扩展论文关于如何蒸馏 The Pile-Deduplicated 的想法来构建 [minipile_density-proportioned](https://huggingface.co/datasets/Marcus2112/minipile_density-proportioned)。

由于 [minipile_density-proportioned](https://huggingface.co/datasets/Marcus2112/minipile_density-proportioned) 被构建为尽可能代表一个更大的、多样化的数据集（The Pile Deduplicated），使用这个蒸馏版本使得预训练高效、多用途，而且更快、更节省资源，对资源受限环境的学习者更加友好。

这是我脚本的原始输出：

---
```
step 0: train loss 10.9707, val loss 10.9712
step 1000: train loss 6.3237, val loss 6.3552
step 2000: train loss 5.9566, val loss 5.9684
step 3000: train loss 5.6662, val loss 5.6960
step 4000: train loss 5.5015, val loss 5.5110
step 5000: train loss 5.3519, val loss 5.3793
step 6000: train loss 5.2336, val loss 5.2582
step 7000: train loss 5.1896, val loss 5.1757
step 8000: train loss 5.0282, val loss 5.0876
step 9000: train loss 5.0095, val loss 5.0130
step 10000: train loss 4.9012, val loss 4.9480
step 11000: train loss 4.8356, val loss 4.8823
step 12000: train loss 4.8116, val loss 4.8242
step 13000: train loss 4.8621, val loss 4.7645
step 14000: train loss 4.8088, val loss 4.7139
step 15000: train loss 4.7179, val loss 4.6632
step 16000: train loss 4.5663, val loss 4.6210
step 17000: train loss 4.5689, val loss 4.5840
step 18000: train loss 4.5564, val loss 4.5486
step 19000: train loss 4.4584, val loss 4.5159
step 20000: train loss 4.4928, val loss 4.4862
step 21000: train loss 4.4235, val loss 4.4589
step 22000: train loss 4.3791, val loss 4.4317
step 23000: train loss 4.3556, val loss 4.4106
step 24000: train loss 4.3154, val loss 4.3879
step 25000: train loss 4.3555, val loss 4.3737
step 26000: train loss 4.3680, val loss 4.3523
step 27000: train loss 4.3849, val loss 4.3361
step 28000: train loss 4.2780, val loss 4.3169
step 29000: train loss 4.3688, val loss 4.3038
step 29268: train loss 4.4642, val loss 4.2959

! Used by Oakland Spaniow on Ste-Rervica


PING WOENET HAS


OTG


May 2, 2018 4:11 AMAY/3/NWR Legal DAITIONART


Kore are cute and diligent. For others this will soon please huge impressions along wouldn't have Patton Magic ever to watch him. This may be an
s grace and sometimes anti-inviting at it. We have to watch this in any moment but let it borrow my mind to the finals, drowning hand (average) contrast. 94, the sale of Christmas
was to break and have looked back along with some
life at something and some sort of rage for this.

27 April 2018 - What am I kind?

Understanding and formal [...]John Farosizzled to Unpluglecka Boxers?

William A. Fur-Reth professor to you in few
cause we know that missed our baby helps raise to add% confidence from bailed, and
silly's, depending on your spouse, else

30 April 2018 2:21 AM 15:

One thing first that has ever progressed to those
and feel
to that once again. In fact, you suffer from embarrassment which 

Starting fine-tuning...
Fine-tuning step 0: train loss 5.5030, val loss 5.2990
Fine-tuning step 1000: train loss 7.6126, val loss 7.6633
Fine-tuning step 2000: train loss 8.0405, val loss 8.1042
Fine-tuning step 3000: train loss 8.3200, val loss 8.3808
Fine-tuning step 4000: train loss 8.6191, val loss 8.6992
Fine-tuning step 4999: train loss 8.7287, val loss 8.8173

Generated text after fine-tuning:
!My work hath yet not warm'd me: he is well:
The blood I drop is rather physical
Than dangerous to me: to Aufidius thus
I will appear, and fight.


LARTIUS:
Now the fair goddess, Fortune,
Fall deep in love with thee; and her great charms
Misguide thy opposers' swords! Bold gentleman,
Prosperity be thy page!


MARCIUS:
Thy friend no less
Than those she placeth highest! So, farewell.


LARTIUS:
Thou worthiest Marcius!
Go, sound thy trumpet in the market-place;
Call thither all the officers o' the town,
Where they shall know our mind: away!


COMINIUS:
Breathe you, my friends: well fought;
we are come off
Like Romans, neither foolish in our stands,
Nor cowardly in retire: believe me, sirs,
We shall be charged again. Whiles we have struck,
By interims and conveying gusts we have heard
The charges of our friends. Ye Roman gods!
Lead their successes as we wish our own, we
```
---

|模型|损失|
|-|-|
| 仅 Tiny-Shakespeare | 训练损失 1.0560，验证损失 1.5007 |
| MiniPile-Density + Tiny-Shakespeare | 训练损失 8.7287，验证损失 8.8173 |

**好的，我们怎么理解这个？**

虽然通过微调得到的损失看起来很糟糕，但实际上并非如此。<br>
结果表明，未预训练的模型在较小的 `tiny-shakespeare` 数据集上明显过拟合。<br>
预训练模型没有过拟合，因此在训练集和验证集上的误差都更高。但它是否仍然在学习有用的东西？<br>
微调是否有意义？让我们来比较。

**仅 Tiny-Shakespeare 的输出：**

```
KING RICHARD II:
Now from sentence come throw and mercy.
Be yonder that I behold to do tower:
'Tis you be longed on our words to much urge.
Never know, with use boody to make and tweMILet him.
To chain the people mother, the rest veril
Let him bring God hence, but be afflied,
Were traitors with one heart the flower days:
The sarch of a companion, dost thou body this heir?
The again all-as this body's world;
That nightly he should be the bal deform'd,
Not latt, but sadisment a medow's is that
Now
```

**MiniPile-Density + Tiny-Shakespeare 的输出：**

```
My work hath yet not warm'd me: he is well:
The blood I drop is rather physical
Than dangerous to me: to Aufidius thus
I will appear, and fight.

LARTIUS:
Now the fair goddess, Fortune,
Fall deep in love with thee; and her great charms
Misguide thy opposers' swords! Bold gentleman,
Prosperity be thy page!

MARCIUS:
Thy friend no less
Than those she placeth highest! So, farewell.

LARTIUS:
Thou worthiest Marcius!
Go, sound thy trumpet in the market-place;
Call thither all the officers o' the town,
Where they shall know our mind: away!

COMINIUS:
Breathe you, my friends: well fought;
we are come off
Like Romans, neither foolish in our stands,
Nor cowardly in retire: believe me, sirs,
We shall be charged again. Whiles we have struck,
By interims and conveying gusts we have heard
The charges of our friends. Ye Roman gods!
```

预训练模型的输出看起来更加连贯和有结构。<br>
还有更复杂的韵律和角色之间的相互引用。

> 我们看到，仅靠微调数据集上的验证损失不能作为模型质量的唯一指标。<br>
> 顺便说一下，不，你不能也不应该期望在微调预训练模型时获得更小的验证损失。

**为什么微调后的模型验证损失更高？**

微调推动从较大数据集获得的丰富理解来纳入较小微调数据集的信息。<br>
换句话说，模型在保留来自较大数据集的更广泛理解的同时，泛化到较小微调数据集的模式和细微差别。<br>
由于预训练模型是在优化泛化而非记忆，微调数据集的验证损失可能更高，但输出更丰富且更符合真实世界的期望。

**为什么不直接用历史文本数据集进行预训练？**

你可以，但预训练应该让模型从广泛的任务和领域中学习通用模式和结构。<br>
如果你在太具体、与微调数据集太相似的数据集上预训练，你就有过度拟合微调数据集特征模式的风险。你会获得更低的验证损失，但模型会更不灵活、对更广泛的世界了解更少，因此更不具创造力。<br>
这就是为什么建议在大型、多样化的数据集上预训练，然后在较小、特定的数据集上微调，以更灵活地利用两者。

**为什么微调时没有选择更高的学习率？**

预训练使用 `3e-4`，微调使用 `2e-5`，以避免所谓的灾难性遗忘（catastrophic forgetting）：<br>
以更高的学习率微调有可能用小数据集的模式覆盖预训练中获得的更广泛知识，使预训练几乎无用。<br>
选择较低的学习率有助于模型在适应微调数据集的同时保留通用知识。<br>
我们可以从上面看到，较小的学习率确实足以让模型捕捉 `tiny-shakespeare` 数据集的主题、风格和结构。

## 练习 4 - 阅读并实现

**目标：** 阅读一些 Transformer 论文并实现一个人们似乎在使用的额外功能或更改。<br>
它是否提高了你的 GPT 的性能？

我将选择[门控线性单元（Gated Linear Units，GLU）\[Shazeer, Noam. 2020\]](https://arxiv.org/abs/2002.05202)。<br>
本质上，我们将用门控机制替换 ReLU 激活函数（activation function），使模型能够更动态/可学习地控制信息流。<br>
模型的其余部分保持不变。

我将这个练习的代码导出到一个单独的脚本中，以便在 [minipile_density-proportioned](https://huggingface.co/datasets/Marcus2112/minipile_density-proportioned) 上更容易运行。<br>
你可以在 [`N007_GPT_Solved_Exercise_Opti.py`](./N007_GPT_Solved_Exercise_Opti.py) 文件中找到该脚本。

预训练模型可以在这里找到：[nanogpt_glu_base](https://huggingface.co/Marcus2112/nanogpt_glu_base)。<br>
微调后的模型可以在这里找到：[nanogpt_glu_shakespeare](https://huggingface.co/Marcus2112/nanogpt_glu_shakespeare)

相同的设置，除了 GLU 激活函数之外一切相同，我们得到：

|模型|损失|
|-|-|
| 仅 Tiny-Shakespeare | 训练损失 1.0560，验证损失 1.5007 |
| MiniPile-Density + Tiny-Shakespeare | 训练损失 8.7287，验证损失 8.8173 |
| MiniPile-Density + Tiny-Shakespeare (GLU) | 训练损失 7.6610，验证损失 7.6660 |

让我们看看使用 GLU 激活函数微调后的生成文本：

```
Was ever man so proud as is this Marcius!

BRUTUS:
He has no equal.

SICINIUS:
When we were chosen tribunes for the people,--

BRUTUS:
Mark'd you his lip and eyes?

SICINIUS:
Nay. but his taunts.

BRUTUS:
Being moved, he will not spare to gird the gods.

SICINIUS:
Be-mock the modest moon.

BRUTUS:
The present wars devour him: he is so valiant.

SICINIUS:
O, true-bred!

BRUTUS:
O, good success, disdains the shadow
Which he treads on at noon: but I do wonder
His insolence can brook to be commanded
Under Cominius.

BRUTUS:
Fame, at the which he aims,
In whom already he's well graced, can not
Better be held nor more attain'd than by
A place below the first:--

BRUTUS:
Fame, at the which he aims,
```

我们有两个角色 Brutus 和 Sicinius 在谈论第三个角色 Marcius。而且，文本实际上是可以理解的：<br>
这段对话反映了 Brutus 和 Sicinius 讨论中对 Marcius 勇气的钦佩与对其傲慢的批评之间的某种张力。

GLU 激活函数似乎提高了模型生成连贯、有结构文本的能力。<br>
通过这种方式，模型似乎学会了（三个）角色之间更复杂的关系，并能够生成更细致的对话。<br>
验证损失没有显著降低，但生成的文本明显更加连贯和有结构。<br>
另外，注意验证和训练损失仍然远高于未预训练模型的损失，这证实了我们在练习 $3$ 中讨论的内容。

**为什么 GLU 有帮助？**

GLU 允许模型可学习地控制信息流。<br>
之前，ReLU 激活函数会简单地将负值归零，这可能过于激进且不可学习。<br>
而 GLU 允许模型学习保留多少负值输入、丢弃多少，即使在周围 token 的上下文中也是如此。<br>
这使得激活更加动态和精确，允许模型学习数据中更复杂的关系和结构。

为上述解答训练的模型：

- 练习 2：
    - [nanogpt_mathematica](https://huggingface.co/Marcus2112/nanogpt_mathematica)
- 练习 3：
    - [nanogpt_base](https://huggingface.co/Marcus2112/nanogpt_base)，
    - [nanogpt_shakespeare](https://huggingface.co/Marcus2112/nanogpt_shakespeare)
- 练习 4：
    - [nanogpt_glu_base](https://huggingface.co/Marcus2112/nanogpt_glu_base)，
    - [nanogpt_glu_shakespeare](https://huggingface.co/Marcus2112/nanogpt_glu_shakespeare)

<center>笔记本由 <a href="https://github.com/mk2112" target="_blank">mk2112</a> 编写。</center>